## 1. Imports & Environment Setup 

In [1]:
# Initial Imports
from pathlib import Path
import json
import pandas as pd
import sys, os
from dotenv import load_dotenv
from mozzarellm.utils.io import load_table
from mozzarellm.utils.screen_context_utils import load_screen_context_json
from mozzarellm.pipeline.bundle_builder import (
    build_evidence_bundles,
    get_or_append_stable_accession,
)
from mozzarellm.clients.llm_api_clients import create_client
from mozzarellm.utils.prompt_factory import (
    make_cluster_analysis_system_prompt,
    make_single_cluster_analysis_user_prompt,
)
from mozzarellm.utils.cluster_utils import build_cluster_id_to_bundle_path

print(sys.path)

['C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\python312.zip', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\DLLs', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64\\Lib', 'C:\\Users\\alexa\\AppData\\Local\\Python\\pythoncore-3.12-64', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv', '', 'c:\\Users\\alexa\\OneDrive - Massachusetts Institute of Technology\\Cheesman Lab\\WORKSPACE\\mozzarellm\\.mozzvenv\\Lib\\site-packages', '__editable__.mozzarellm-0.2.0.finder.__path_hook__']


In [2]:
def _find_repo_root(start_dir: Path) -> Path:
    start_dir = start_dir.resolve()
    for p in [start_dir] + list(start_dir.parents):
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find pyproject.toml starting from {start_dir}")


# Use repo root for sys.path and for locating .env, regardless of notebook working directory (less brittle).
repo_root = _find_repo_root(Path(os.getcwd()))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# Load secrets/config from .env at the repo root (if present)
print("Does .env exist in repo root?", (repo_root / ".env").exists())
load_dotenv(dotenv_path=repo_root / ".env")
print("Repo root:", repo_root)
print("Current working directory:", os.getcwd())

Does .env exist in repo root? True
Repo root: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm
Current working directory: c:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\interface


## 2. Screen Context Configuration

Complete the `screen_context.json` file with the relevant information for your screen. Feel free to adjust the template as needed beyond the required fields. The top level must be a dictionary, and all fields must be strings. Keep the file in the same directory as the notebook. This cell will check for the file and raise an error if it is not found or not well formatted. 

In [3]:
# set screen name
SCREEN_NAME = "test_with_CoT"
SCREEN_CONTEXT_PATH = Path("screen_context.json")  # or change to your custom filename.
screen_ctx = load_screen_context_json(SCREEN_CONTEXT_PATH)
# should throw error if file doesn't exist or if you forgot to remove the TODO!

print("Loaded screen_context.json:", isinstance(screen_ctx, dict))
keys = list(screen_ctx.keys())
print(f"Top-level keys ({len(keys)}): {keys}")

Loaded screen_context.json: True
Top-level keys (10): ['assay_type', 'target_phenotype', 'organism', 'cell_line_or_system', 'perturbation', 'readout', 'clustering', 'controls', 'provenance', 'notes']


## 3. Load & Standardize Cluster Table 
### **3.1** Initial Load
Set parameters and check that the original cluster table loads.


In [4]:
#### 3.1.1 Point this to your cluster/gene table (CSV/TSV/TXT/XLSX) ####
default_cluster_table = (
    repo_root / "examples" / "ops" / "funk_2022.csv"
)  # NOTE: using funk_2022.csv for development
CLUSTER_TABLE_PATH = Path(default_cluster_table)

#### 3.1.2 Optional: ####
sep = None
# Useful for .txt files. If the table loads as one big column (or in an otherwise wonky fashion), the delimiter is likely mismatched.
# By default pandas ASSUMES the delimiter (value separator) based on file extension
# (typically .csv → ',', .tsv → '\t'). Common overrides are ',', '|', '\t', or ';'.

sheet_name = None
# For Excel only (e.g. 0 or 'Sheet1'); ignored for CSV/TSV
#########################

CLUSTER_DF = load_table(CLUSTER_TABLE_PATH, sep=sep, sheet_name=sheet_name)
print("Loaded:", str(CLUSTER_TABLE_PATH))
print("Shape:", CLUSTER_DF.shape)
print("Columns:\n", list(CLUSTER_DF.columns))
display(CLUSTER_DF.head())

Loaded: C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\examples\ops\funk_2022.csv
Shape: (164, 5)
Columns:
 ['gene_symbol', 'cluster', 'up_features', 'down_features', 'phenotypic_strength']


,gene_symbol,cluster,up_features,down_features,phenotypic_strength
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299


### **3.2** Gene to stable accession ID mapping

If you do not have stable accession IDs in your cluster table, we will make a best effort attempt to assign them based on your provided gene symbols (NOTE: the `get_accession_from_gene_symbol` method is available for review in `clients/uniprot_api_client.py`).

If you have a gene to stable accession IDs maping available separately from your cluster table, you can provide this table, and we will append the accession IDs to your cluster table. 

A csv copy of the assigned or appended gene to stable accession mapping will be provided for review in the `interface/output/` directory.

In [5]:
# set cluster id column name or keep default
CLUSTER_ID_COLUMN = "cluster"
# set gene column name or keep default
GENE_COLUMN = "gene_symbol"

# If you have a stable accession column already in your cluster table, set it here.
# If no stable accession column, leave as None and we'll handle it for you.
STABLE_ACCESSION_COLUMN = None  # or the name of your stable accession column

# List of per-gene feature columns that would be useful to determine overlaps for within a cluster.
FEATURE_COLUMNS = [
    "up_features",
    "down_features",
]

# Set organism id or keep default
ORGANISM_ID = 9606  # human

if STABLE_ACCESSION_COLUMN is None:
    ACC_CLUSTER_DF = get_or_append_stable_accession(
        screen_name=SCREEN_NAME,
        cluster_df=CLUSTER_DF,
        gene_column=GENE_COLUMN,
        organism_id=ORGANISM_ID,
        # SET WARNING SETTING: default is to ignore warnings for unreviewed entries
        warn_on_fallback=False,
        # IF ADDING A SEPERATE TABLE, SET THESE:
        accession_table=None,  # or Path("path/to/accession_table.csv")
        accession_col=None,  # or the name of your stable accession column in the seperate table
        accession_table_gene_col=None,  # or the name of your gene column in the seperate table
        accession_table_sheetname=None,  # or the sheet name for the seperate table if it's an excel file
        accession_table_sep=None,  # or the delimiter for the seperate table (use if you encounter encoding issues)
    )

# NOTE: Currently, running this cell will overwrite the last .csv output.
# TODO: (later) add option to prevent overwriting with a timestamp in filename. [just a hassle during iteration]
ACC_CLUSTER_DF

Assigned accession numbers to cluster df. Saved to:  output\test_with_CoT_analysis\intermediates\assigned_accession_cluster_df.csv


,gene_symbol,cluster,up_features,down_features,phenotypic_strength,accession
0,AATF,21,"interphase_cell_correlation_dapi_tubulin,inter...","interphase_nucleus_area,interphase_cell_area,i...",669/5299,Q9NY61
1,ABT1,21,"interphase_nucleus_gh2ax_mean,interphase_nucle...","interphase_nucleus_area,interphase_cell_area,i...",560/5299,Q9ULW3
2,BMS1,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",446/5299,Q14692
3,BYSL,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",524/5299,Q13895
4,C1orf131,21,"interphase_nucleus_dapi_mean,interphase_cell_c...","interphase_nucleus_area,interphase_cell_area,i...",343/5299,Q8NDD1
...,...,...,...,...,...,...
159,nontargeting_5_2,99,NaN,NaN,5203/5299,NON_TARGETING_CONTROL
160,nontargeting_61_0,99,NaN,NaN,5038/5299,NON_TARGETING_CONTROL
161,nontargeting_6_1,99,NaN,interphase_cell_phalloidin_mean,5237/5299,NON_TARGETING_CONTROL
162,nontargeting_8_2,99,NaN,NaN,5271/5299,NON_TARGETING_CONTROL


## 4. Create Per-Cluster Evidence Bundles

Bundles will be saved to the `interface/output` directory. Additionally, each chunk has retrieved annotations and evidence logged in a corresponding csv file within the same directory. These can be accessed via File Explorer and viewed in Excel or Google Sheets.

In [6]:
build_evidence_bundles(
    screen_name=SCREEN_NAME,
    acc_cluster_df=ACC_CLUSTER_DF,  # cluster table must have accession numbers
    gene_column=GENE_COLUMN,
    cluster_id_column=CLUSTER_ID_COLUMN,
    stable_accession_col=STABLE_ACCESSION_COLUMN or "accession",
    feature_columns=FEATURE_COLUMNS,
    use_retrieval=False,  # false for now; true for future use
    knowledge_dir=None,
    top_k=5,
)

Using gene column: gene_symbol
Processing 7 clusters
Processing cluster 21


Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_21__bundle.json
Processing cluster 37
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_37__bundle.json
Processing cluster 121
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_121__bundle.json
Processing cluster 149
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_149__bundle.json
Processing cluster 167
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_167__bundle.json
Processing cluster 197
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_197__bundle.json
Processing cluster 99
Saved bundle to output\test_with_CoT_analysis\test_with_CoT_evidence_bundles\test_with_CoT__cluster_99__bundle.json


C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\mozzarellm\clients\uniprot_api_client.py:253: UserWarning: 16 accession(s) found but lack FUNCTION annotations: ['Q9BVI4', 'Q5JTH9', 'Q9BSC4', 'Q3SX64', 'Q66K80']...
  warnings.warn(
C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\mozzarellm\clients\uniprot_api_client.py:253: UserWarning: 6 accession(s) found but lack FUNCTION annotations: ['Q96PV6', 'Q96BW1', 'P28290', 'Q8WU58', 'A5PLN9']...
  warnings.warn(
C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\mozzarellm\clients\uniprot_api_client.py:253: UserWarning: 7 accession(s) found but lack FUNCTION annotations: ['Q9NT22', 'Q0D2H9', 'Q96KN3', 'Q6P1L5', 'Q96PV6']...
  warnings.warn(
C:\Users\alexa\OneDrive - Massachusetts Institute of Technology\Cheesman Lab\WORKSPACE\mozzarellm\mozzarellm\clients\uniprot_api_client.py:253: UserWarning: 11

In [7]:
# build cluster ID to evidence bundle path mapping
BUNDLES_DIR = Path(
    f"output/{SCREEN_NAME}_analysis/{SCREEN_NAME}_evidence_bundles"
)  # check that this aligns with the output dir above
CLUSTER_ID_TO_BUNDLE_PATH_MAP = build_cluster_id_to_bundle_path(
    BUNDLES_DIR, screen_name=SCREEN_NAME
)
CLUSTER_ID_TO_BUNDLE_PATH_MAP

{'121': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_121__bundle.json'),
 '149': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_149__bundle.json'),
 '167': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_167__bundle.json'),
 '197': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_197__bundle.json'),
 '21': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_21__bundle.json'),
 '37': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_37__bundle.json'),
 '99': WindowsPath('output/test_with_CoT_analysis/test_with_CoT_evidence_bundles/test_with_CoT__cluster_99__bundle.json')}

## 5. LLM Analysis (Initial Hypothesis Formation)

### **5.1** Prompt Construction


In [8]:
SYSTEM_PROMPT = make_cluster_analysis_system_prompt(
    screen_name=SCREEN_NAME,
    screen_context_path=Path("screen_context.json"),
    template_path=None,  # or your custom .md or .txt file to override the default template
    template_string=None,
    CoT_mode=True,
)
print(SYSTEM_PROMPT)
# NOTE: we will be appending ""\n\n The following experimental context is provided: " + SCREEN_CONTEXT + "\n\n"" to any custom template you provide


MISSION: Functional genomics experiments cluster genes by phenotypic similarity. Your goal is to:
1. Identify the dominant biological pathway that explains why these genes cluster together
2. Classify ALL genes relative to this pathway (ESTABLISHED / UNCHARACTERIZED / NOVEL_ROLE)
3. Prioritize understudied genes (UNCHARACTERIZED and NOVEL_ROLE) for follow-up experiments

The pathway is not the end goal - it's the lens for discovering which genes merit investigation.



STEP 1 - PATHWAY HYPOTHESIS (2-3 candidates):
- Review gene annotations
- List 2-3 candidate pathways with supporting genes
- Note which annotations support each hypothesis

STEP 2 - PATHWAY SELECTION:
- Select dominant pathway based on:
  * Number of established genes with direct roles
  * Coherence of functional relationships
  * Quality of supporting evidence (prioritize high-relevance snippets)
- Assign confidence level using the following criteria: 
ASSESSING PATHWAY CONFIDENCE:

Once you have identified a candidat

In [9]:
# FOR SINGLE CLUSTER QUERY (use this as a test before running the full analysis)
# Pick a cluster ID (string keys)
CLUSTER_ID = "121"  # change me
USER_PROMPT = make_single_cluster_analysis_user_prompt(
    CLUSTER_ID, SCREEN_NAME, CLUSTER_ID_TO_BUNDLE_PATH_MAP
)
print(
    json.dumps(
        json.loads(Path(CLUSTER_ID_TO_BUNDLE_PATH_MAP[str(CLUSTER_ID)]).read_text()), indent=2
    )
)

{
  "screen_name": "test_with_CoT",
  "cluster_id": "121",
  "cluster_genes": [
    {
      "gene_symbol": "CCDC174",
      "up_features": "interphase_nucleus_area,interphase_cell_dapi_mass_displacement,interphase_cell_correlation_dapi_tubulin,interphase_nucleus_eccentricity,interphase_cell_area",
      "down_features": "interphase_nucleus_form_factor,interphase_nucleus_dapi_mean,interphase_nucleus_correlation_dapi_gh2ax",
      "phenotypic_strength": "2240/5299",
      "UniProt_functional_annotation": "Probably involved in neuronal development",
      "accession": "Q6PII3"
    },
    {
      "gene_symbol": "DDA1",
      "up_features": "interphase_cell_area,interphase_cell_correlation_dapi_tubulin,interphase_nucleus_area",
      "down_features": "interphase_nucleus_correlation_dapi_gh2ax,interphase_nucleus_dapi_mean,interphase_nucleus_form_factor,interphase_nucleus_solidity",
      "phenotypic_strength": "2006/5299",
      "UniProt_functional_annotation": "Functions as a component of n

### **5.2** Query the LLM

In [10]:
# init client
LLM_client = create_client(
    model="claude-sonnet-4-6",  # or your preferred model
    api_key=os.getenv("ANTHROPIC_API_KEY"),  # or your preferred API key
)

print("Client initialized.")

Client initialized.


In [12]:
############################ SINGLE CLUSTER QUERY ################################
response_text, error_msg = LLM_client.query(
    max_retries=3,
    batch=False,
    system_prompt=SYSTEM_PROMPT,
    user_prompt=USER_PROMPT,
)

if error_msg:
    raise RuntimeError(error_msg)

# Display in notebook
print(response_text)

# Save response for later inspection
out_path = Path(
    f"output/{SCREEN_NAME}_analysis/phase1_single_cluster_LLM_analysis/{CLUSTER_ID}_single_cluster_analysis_response.txt"
)
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(response_text, encoding="utf-8")
print(f"Saved response to: {out_path}")

# Step-by-Step Analysis

## STEP 1 - PATHWAY HYPOTHESIS

**Candidate 1: Transcriptional Regulation / Gene Expression Control**
- Supporting genes: MYC, MAX, GABPA, GABPB1, SP2, FOXN1, ZBTB11, ZMYND8, SON, ILF3, HNRNPM
- MYC-MAX complex is a canonical transcription factor pair; GABPA/GABPB1 form the GA-binding protein complex; SP2 is a GC-box transcription factor; ZMYND8 is a chromatin reader/transcriptional coregulator

**Candidate 2: Chromatin Remodeling / Epigenetic Regulation**
- Supporting genes: SETD2 (H3K36me3 methyltransferase), ZMYND8 (chromatin reader), N6AMT1 (H4K12me1), MAX (chromatin remodeling recruitment), KEAP1 (ubiquitin ligase targeting chromatin factors)
- Strong epigenetic theme but fewer dedicated members

**Candidate 3: MYC Transcriptional Network**
- Supporting genes: MYC, MAX (direct MYC partner), GABPA/GABPB1 (ETS family, co-regulates MYC targets), SETD2 (H3K36me3 at MYC targets), ZMYND8 (P-TEFb recruitment, transcription elongation), SON (splicing cofactor for 

In [ ]:
########################### ITERATIVE QUERY FOR ALL CLUSTERS ################################
for cluster_id in CLUSTER_ID_TO_BUNDLE_PATH_MAP.keys():
    print(f"Processing cluster {cluster_id}")
    USER_PROMPT = make_single_cluster_analysis_user_prompt(
        cluster_id, SCREEN_NAME, CLUSTER_ID_TO_BUNDLE_PATH_MAP
    )

    response_text, error_msg = LLM_client.query(
        max_retries=3,
        batch=False,
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
    )

    if error_msg:
        raise RuntimeError(error_msg)

    # Save response for later inspection
    out_path = Path(
        f"output/{SCREEN_NAME}_analysis/phase1_iterative_cluster_LLM_analysis/{cluster_id}_single_cluster_analysis_response.txt"
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(response_text, encoding="utf-8")
    print(f"Done! Saved response to: {out_path}")

In [ ]:
################################# BATCH QUERY ##################################
# # NOTE: Uncomment to run batch query. Only useful if you have a large number of clusters to analyze.
# LLM_client.query(
#     max_retries=3,
#     batch=True,
#     screen_name="test",
#     system_prompt=SYSTEM_PROMPT,
#     cluster_to_prompt_map=CLUSTER_ID_TO_BUNDLE_PATH_MAP,
# )

Long processing time? Retrieve results using the saved batch ID by uncommenting and running the following code:

<sub>*NOTE: Batch request results must be retrieved within 24 hours of submission.* </sub>

In [ ]:
# import anthropic
# import time

# # Use the raw Anthropic SDK client directly
# client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

# batch_id = "msgbatch_01AKRRYWgpbZmXrSDUXnc3s2" # Replace with your batch ID
# screen_name = "your_screen_name"  # Define this

# # Polling for message batch completion
# while True:
#     message_batch = client.messages.batches.retrieve(batch_id)
#     if message_batch.processing_status == "ended":
#         break
#     print(f"Batch {batch_id} is still processing...")
#     time.sleep(60)

# # Stream results file in memory-efficient chunks, processing one at a time
# for result in client.messages.batches.results(batch_id):
#     match result.result.type:
#         case "succeeded":
#             # Extract the actual message content
#             message_content = result.result.message.content[0].text

#             path = Path(
#                 f"output/{screen_name}_analysis/phase1_batch_cluster_LLM_analysis/{result.custom_id}_analysis_response.txt"
#             )
#             path.parent.mkdir(parents=True, exist_ok=True)
#             path.write_text(message_content, encoding="utf-8")
#             print(f"{result.custom_id} succeeded. Saving response to {path}")
#         case "errored":
#             if result.result.error.type == "invalid_request":
#                 print(f"Validation error {result.custom_id}: {result.result.error.type}")
#             else:
#                 print(f"Server error {result.custom_id}: {result.result.error.type}")
#         case "expired":
#             print(f"Request expired {result.custom_id}")